# Usar Q-table entrenada para generar String Art

Carga la Q-table creada en el notebook de entrenamiento y genera una imagen nueva usando el agente mixto.

In [1]:
# Si estás en Colab y falta alguna librería, ejecuta esta celda.
# !pip install gymnasium opencv-python matplotlib numpy pandas

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import json


In [2]:
class EntornoHilograma(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self, ruta_imagen, num_clavos=120, resolucion=500, oscuridad_linea=20, max_lineas=1500, pixeles_linea=None):
        super().__init__()
        self.num_clavos = num_clavos
        self.res = resolucion
        self.centro = resolucion // 2
        self.radio = self.centro - 10
        self.oscuridad_linea = oscuridad_linea
        self.max_lineas = max_lineas
        self.ruta_imagen = ruta_imagen
        self.action_space = spaces.Discrete(self.num_clavos)
        self.observation_space = spaces.Box(low=0, high=255, shape=(self.res, self.res), dtype=np.int32)
        self.clavos = self._calcular_posiciones_clavos()
        self.pixeles_linea = pixeles_linea if pixeles_linea is not None else self._precalcular_pixeles_linea()
        self._preprocesar_imagen_objetivo()

    def _calcular_posiciones_clavos(self):
        angulos = np.linspace(0, 2 * np.pi, self.num_clavos, endpoint=False)
        x = self.centro + self.radio * np.cos(angulos)
        y = self.centro + self.radio * np.sin(angulos)
        return np.column_stack((x, y)).astype(np.int32)

    def _precalcular_pixeles_linea(self):
        print(f"[Entorno] Precalculando píxeles para {self.num_clavos * (self.num_clavos - 1) // 2} líneas...")
        pixeles_linea = {}
        n = self.num_clavos
        for i in range(n):
            for j in range(i + 1, n):
                mascara = np.zeros((self.res, self.res), dtype=np.uint8)
                cv2.line(mascara, tuple(self.clavos[i]), tuple(self.clavos[j]), 1, 1)
                ys, xs = np.nonzero(mascara)
                pixeles_linea[(i, j)] = (xs, ys)
                pixeles_linea[(j, i)] = (xs, ys)
        print("[Entorno] Precálculo terminado.")
        return pixeles_linea

    def _preprocesar_imagen_objetivo(self):
        img = cv2.imread(str(self.ruta_imagen), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"No se pudo cargar la imagen desde: {self.ruta_imagen}")
        h, w = img.shape
        dim_min = min(h, w)
        inicio_h = (h - dim_min) // 2
        inicio_w = (w - dim_min) // 2
        img_recortada = img[inicio_h:inicio_h + dim_min, inicio_w:inicio_w + dim_min]
        img_redimensionada = cv2.resize(img_recortada, (self.res, self.res))
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img_redimensionada)
        self.imagen_objetivo = 255 - img_clahe
        mascara = np.zeros((self.res, self.res), dtype=np.uint8)
        cv2.circle(mascara, (self.centro, self.centro), self.radio, 255, -1)
        self.imagen_objetivo = cv2.bitwise_and(self.imagen_objetivo, self.imagen_objetivo, mask=mascara)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.acumulador = np.zeros((self.res, self.res), dtype=np.int32)
        self.contador_lineas = 0
        self.clavo_actual = int(self.np_random.integers(0, self.num_clavos))
        return self.acumulador, {"clavo_actual": self.clavo_actual}

    def ganancia_linea(self, clavo_origen, clavo_destino):
        if clavo_origen == clavo_destino:
            return -100.0
        xs, ys = self.pixeles_linea[(clavo_origen, clavo_destino)]
        valores_objetivo = self.imagen_objetivo[ys, xs]
        valores_acumulador = self.acumulador[ys, xs]
        return float(np.sum(np.maximum(0, valores_objetivo - valores_acumulador)))

    def ganancias_desde_clavo_actual(self):
        ganancias = np.array([self.ganancia_linea(self.clavo_actual, j) for j in range(self.num_clavos)], dtype=float)
        ganancias[self.clavo_actual] = -np.inf
        return ganancias

    def step(self, action):
        siguiente_clavo = int(action)
        if siguiente_clavo == self.clavo_actual:
            info = {"clavo_actual": self.clavo_actual, "error": "Mismo clavo"}
            return self.acumulador, -100.0, False, False, info
        xs, ys = self.pixeles_linea[(self.clavo_actual, siguiente_clavo)]
        recompensa = self.ganancia_linea(self.clavo_actual, siguiente_clavo)
        self.acumulador[ys, xs] += self.oscuridad_linea
        self.clavo_actual = siguiente_clavo
        self.contador_lineas += 1
        truncated = self.contador_lineas >= self.max_lineas
        info = {"clavo_actual": self.clavo_actual, "linea_actual": self.contador_lineas}
        return self.acumulador, recompensa, False, truncated, info

    def render(self, mode="human"):
        lienzo = np.clip(255 - self.acumulador, 0, 255).astype(np.uint8)
        if mode == "rgb_array":
            return lienzo
        plt.figure(figsize=(6, 6))
        plt.imshow(lienzo, cmap="gray", vmin=0, vmax=255)
        plt.axis("off")
        plt.show()
        return lienzo


In [3]:
class AgenteMixtoQLearning:
    """Agente mixto: ganancia inmediata + novedad + salto + Q-table aprendida."""
    nombre = "Mixto con Q-table"

    def __init__(self, num_clavos, peso_ganancia=0.60, peso_novedad=0.15, peso_salto=0.10, peso_q=0.15,
                 alpha=0.12, gamma=0.80, epsilon=0.08, seed=42, q_table=None):
        self.num_clavos = num_clavos
        self.peso_ganancia = peso_ganancia
        self.peso_novedad = peso_novedad
        self.peso_salto = peso_salto
        self.peso_q = peso_q
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.rng = np.random.default_rng(seed)
        self.visitas = np.zeros(num_clavos, dtype=float)
        self.q_table = np.zeros((num_clavos, num_clavos), dtype=np.float32) if q_table is None else q_table.astype(np.float32)

    def _normalizar_q(self, origen):
        q = self.q_table[origen].astype(float).copy()
        q[origen] = -np.inf
        validas = np.isfinite(q)
        if not np.any(validas):
            return np.zeros(self.num_clavos)
        q_valid = q[validas]
        q_min, q_max = np.min(q_valid), np.max(q_valid)
        q_norm = np.zeros(self.num_clavos)
        if q_max > q_min:
            q_norm[validas] = (q_valid - q_min) / (q_max - q_min)
        return q_norm

    def seleccionar_accion(self, entorno):
        origen = entorno.clavo_actual
        ganancias = entorno.ganancias_desde_clavo_actual()
        validas = np.isfinite(ganancias)
        acciones_validas = np.where(validas)[0]
        if self.rng.random() < self.epsilon:
            return int(self.rng.choice(acciones_validas))

        g = np.zeros_like(ganancias, dtype=float)
        max_g = np.max(ganancias[validas]) if np.any(validas) else 1.0
        if max_g <= 0:
            max_g = 1.0
        g[validas] = ganancias[validas] / max_g
        novedad = 1.0 / (1.0 + self.visitas)
        dist_circular = np.array([min(abs(j - origen), self.num_clavos - abs(j - origen)) for j in range(self.num_clavos)], dtype=float)
        salto = dist_circular / (self.num_clavos / 2)
        q_norm = self._normalizar_q(origen)
        score = (self.peso_ganancia*g + self.peso_novedad*novedad + self.peso_salto*salto + self.peso_q*q_norm)
        score[~validas] = -np.inf
        return int(np.argmax(score))

    def actualizar(self, origen, accion, recompensa, nuevo_estado):
        r = recompensa / (abs(recompensa) + 10000.0)  # normalización estable 0..1 aprox.
        mejor_futuro = np.max(np.delete(self.q_table[nuevo_estado], nuevo_estado))
        td = r + self.gamma * mejor_futuro - self.q_table[origen, accion]
        self.q_table[origen, accion] += self.alpha * td
        self.visitas[accion] += 1


In [4]:
def cargar_q_table(ruta_qtable):
    data = np.load(ruta_qtable, allow_pickle=True)
    q_table = data['q_table'].astype(np.float32)
    meta = {k: data[k].item() if data[k].shape == () else data[k] for k in data.files if k != 'q_table'}
    return q_table, meta


def generar_con_q_table(ruta_imagen, ruta_qtable, configuracion=None, carpeta_salida='resultado_con_qtable'):
    carpeta_salida = Path(carpeta_salida)
    carpeta_salida.mkdir(exist_ok=True)
    q_table, meta = cargar_q_table(ruta_qtable)
    if configuracion is None:
        configuracion = {
            'num_clavos': int(meta.get('num_clavos', q_table.shape[0])),
            'resolucion': int(meta.get('resolucion', 500)),
            'oscuridad_linea': int(meta.get('oscuridad_linea', 20)),
            'max_lineas': int(meta.get('max_lineas', 1500)),
        }
    if configuracion['num_clavos'] != q_table.shape[0]:
        raise ValueError(f"La Q-table tiene {q_table.shape[0]} clavos, pero configuracion usa {configuracion['num_clavos']}.")

    entorno = EntornoHilograma(ruta_imagen, **configuracion)
    agente = AgenteMixtoQLearning(num_clavos=configuracion['num_clavos'], q_table=q_table,
                                  peso_ganancia=0.70, peso_novedad=0.05, peso_salto=0.05, peso_q=0.20,
                                  epsilon=0.0)
    _, _ = entorno.reset(seed=42)
    recompensa_total = 0.0
    terminado = truncado = False
    while not (terminado or truncado):
        accion = agente.seleccionar_accion(entorno)
        _, recompensa, terminado, truncado, info = entorno.step(accion)
        recompensa_total += recompensa
        if info['linea_actual'] % max(1, configuracion['max_lineas']//5) == 0:
            print(f"línea {info['linea_actual']} | recompensa acumulada {recompensa_total:.2f}")
    imagen_final = entorno.render(mode='rgb_array')
    salida = carpeta_salida / 'string_art_con_qtable.png'
    cv2.imwrite(str(salida), imagen_final)
    print('Imagen guardada en:', salida)
    return imagen_final, salida


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
# Rutas de entrada
ruta_qtable = '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/q_table_mixto.npz'  # sube aquí tu Q-table o coloca ruta de Drive
ruta_imagen = '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/gato.jpg'

# Debe coincidir con num_clavos de la Q-table. Puedes subir max_lineas si quieres más detalle.
configuracion = None  # usa la configuración guardada dentro de la Q-table

In [8]:
imagen_final, salida = generar_con_q_table(ruta_imagen, ruta_qtable, configuracion=configuracion)

plt.figure(figsize=(7,7))
plt.imshow(imagen_final, cmap='gray', vmin=0, vmax=255)
plt.axis('off')
plt.show()

[Entorno] Precalculando píxeles para 44850 líneas...


KeyboardInterrupt: 

In [ ]:
# Descargar resultado en Google Colab
try:
    from google.colab import files
    files.download(str(salida))
except Exception:
    print('Si no estás en Colab, descarga manualmente:', salida)

## ¿Será rápido?

Sí, será más rápido que entrenar desde cero con muchas imágenes porque ya no repite el aprendizaje. Pero todavía debe procesar la imagen nueva y evaluar líneas. Para máxima velocidad usa menos `num_clavos`, menor `resolucion` y menos `max_lineas`.